<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


#  SVM (Support Vector Machines)


Estimated time needed: **15** minutes
    

## Objectives

After completing this lab you will be able to:

* Use scikit-learn to Support Vector Machine to classify


En este cuaderno, utilizará SVM (máquinas de vectores de soporte) para crear y entrenar un modelo utilizando registros de células humanas y clasificar las células según sean benignas o malignas.

SVM funciona asignando datos a un espacio de características de alta dimensión para que los puntos de datos se puedan categorizar, incluso cuando los datos no se puedan separar linealmente de otra manera. Se encuentra un separador entre las categorías y luego los datos se transforman de tal manera que el separador se pueda dibujar como un hiperplano. A continuación, se pueden utilizar las características de los nuevos datos para predecir el grupo al que debe pertenecer un nuevo registro.


<h1>Table of contents</h1>

<div class="alert alert-block alert-info" style="margin-top: 20px">
    <ol>
        <li><a href="#load_dataset">Load the Cancer data</a></li>
        <li><a href="#modeling">Modeling</a></li>
        <li><a href="#evaluation">Evaluation</a></li>
        <li><a href="#practice">Practice</a></li>
    </ol>
</div>
<br>
<hr>


In [ ]:
!pip install scikit-learn
!pip install matplotlib
!pip install pandas 
!pip install numpy 
%matplotlib inline

In [ ]:
import pandas as pd
import pylab as pl
import numpy as np
import scipy.optimize as opt
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
%matplotlib inline 
import matplotlib.pyplot as plt

<h2 id="load_dataset">Cargar los datos de cáncer</h2>
El ejemplo se basa en un conjunto de datos que está disponible públicamente en el Repositorio de aprendizaje automático de la UCI (Asuncion y Newman, 2007) [http://mlearn.ics.uci.edu/MLRepository.html]. El conjunto de datos consta de varios cientos de registros de muestras de células humanas, cada uno de los cuales contiene los valores de un conjunto de características celulares. Los campos de cada registro son:

<br>
<br>

|Field name|Description|
|--- |--- |
|ID|Clump thickness|
|Clump|Clump thickness|
|UnifSize|Uniformity of cell size|
|UnifShape|Uniformity of cell shape|
|MargAdh|Marginal adhesion|
|SingEpiSize|Single epithelial cell size|
|BareNuc|Bare nuclei|
|BlandChrom|Bland chromatin|
|NormNucl|Normal nucleoli|
|Mit|Mitoses|
|Class|Benign or malignant|

<br>
<br>

Para los fines de este ejemplo, utilizamos un conjunto de datos que tiene una cantidad relativamente pequeña de predictores en cada registro. Para descargar los datos, utilizaremos `!wget` para descargarlos desde IBM Object Storage.


In [ ]:
#Click here and press Shift+Enter
!wget -O cell_samples.csv https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%203/data/cell_samples.csv

## Load Data From CSV File  


In [ ]:
cell_df = pd.read_csv("cell_samples.csv")
cell_df.head()

El campo ID contiene los identificadores del paciente. Las características de las muestras de células de cada paciente se incluyen en los campos Clump a Mit. Los valores se clasifican de 1 a 10, siendo 1 el más cercano a benigno.

El campo Class contiene el diagnóstico, confirmado por procedimientos médicos separados, de si las muestras son benignas (valor = 2) o malignas (valor = 4).

Veamos la distribución de las clases en función del grosor del Clump y la uniformidad del tamaño celular:


In [ ]:
ax = cell_df[cell_df['Class'] == 4][0:50].plot(kind='scatter', x='Clump', y='UnifSize', color='DarkBlue', label='malignant');
cell_df[cell_df['Class'] == 2][0:50].plot(kind='scatter', x='Clump', y='UnifSize', color='Yellow', label='benign', ax=ax);
plt.show()

## Data pre-processing and selection


Veamos primero los tipos de datos de las columnas:


In [ ]:
cell_df.dtypes

Parece que la columna __BareNuc__ incluye algunos valores que no son numéricos. Podemos eliminar esas filas:


In [ ]:
cell_df = cell_df[pd.to_numeric(cell_df['BareNuc'], errors='coerce').notnull()]
cell_df['BareNuc'] = cell_df['BareNuc'].astype('int')
cell_df.dtypes

In [ ]:
feature_df = cell_df[['Clump', 'UnifSize', 'UnifShape', 'MargAdh', 'SingEpiSize', 'BareNuc', 'BlandChrom', 'NormNucl', 'Mit']]
X = np.asarray(feature_df)
X[0:5]

We want the model to predict the value of Class (that is, benign (=2) or malignant (=4)).


In [ ]:
y = np.asarray(cell_df['Class'])
y [0:5]

## Train/Test dataset


We split our dataset into train and test set:


In [ ]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=4)
print ('Train set:', X_train.shape,  y_train.shape)
print ('Test set:', X_test.shape,  y_test.shape)

<h2 id="modeling">Modeling (SVM with Scikit-learn)</h2>


El algoritmo SVM ofrece una selección de funciones de kernel para realizar su procesamiento. Básicamente, la transformación de datos en un espacio de mayor dimensión se denomina kernelling. La función matemática utilizada para la transformación se conoce como función de kernel y puede ser de diferentes tipos, como:

    1.Linear
    2.Polynomial
    3.Radial basis function (RBF)
    4.Sigmoid
Cada una de estas funciones tiene sus características, sus pros y sus contras, y su ecuación, pero como no hay una manera sencilla de saber qué función funciona mejor con un conjunto de datos determinado, normalmente elegimos distintas funciones una por una y comparamos los resultados. Para este laboratorio, utilizaremos la función de base radial (RBF) predeterminada.


In [ ]:
from sklearn import svm
clf = svm.SVC(kernel='rbf')
clf.fit(X_train, y_train) 

Una vez ajustado, el modelo puede utilizarse para predecir nuevos valores:


In [ ]:
yhat = clf.predict(X_test)
yhat [0:5]

<h2 id="evaluation">Evaluation</h2>


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import itertools

In [ ]:
def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    print(cm)

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

In [ ]:
# Compute confusion matrix
cnf_matrix = confusion_matrix(y_test, yhat, labels=[2,4])
np.set_printoptions(precision=2)

print (classification_report(y_test, yhat))

# Plot non-normalized confusion matrix
plt.figure()
plot_confusion_matrix(cnf_matrix, classes=['Benign(2)','Malignant(4)'],normalize= False,  title='Confusion matrix')

También puedes utilizar fácilmente __f1_score__ de la biblioteca sklearn:

In [ ]:
from sklearn.metrics import f1_score
f1_score(y_test, yhat, average='weighted') 

Probemos el índice Jaccard para comprobar la precisión:

In [ ]:
from sklearn.metrics import jaccard_score
jaccard_score(y_test, yhat,pos_label=2)

<h2 id="practice">Practice</h2>
¿Puedes reconstruir el modelo, pero esta vez con un  __linear__ kernel? Puedes usar la opción __kernel='linear'__ cuando defines el svm. ¿Cómo cambia la precisión con la nueva función kernel?


In [ ]:
# write your code here


<details><summary>Click here for the solution</summary>

```python
clf2 = svm.SVC(kernel='linear')
clf2.fit(X_train, y_train) 
yhat2 = clf2.predict(X_test)
print("Avg F1-score: %.4f" % f1_score(y_test, yhat2, average='weighted'))
print("Jaccard score: %.4f" % jaccard_score(y_test, yhat2,pos_label=2))

```

</details>



### Thank you for completing this lab!


## Author

Saeed Aghabozorgi


### Other Contributors

<a href="https://www.linkedin.com/in/joseph-s-50398b136/" target="_blank">Joseph Santarcangelo</a>

## <h3 align="center"> © IBM Corporation 2020. All rights reserved. <h3/>

<!--
## Change Log


|  Date (YYYY-MM-DD) |  Version | Changed By  |  Change Description |
|---|---|---|---|
| 2021-01-21  | 2.2  | Lakshmi  |  Updated sklearn library |
| 2020-11-03  | 2.1  | Lakshmi  |  Updated URL of csv |
| 2020-08-27  | 2.0  | Lavanya  |  Moved lab to course repo in GitLab |
|   |   |   |   |
|   |   |   |   |
--!>


